In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "notebook_setup.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_setup import get_dataset_path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from sklearn.manifold import TSNE

# =============================================================================
# =============================================================================
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20

mat_path = get_dataset_path("CETL_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat = data["processed_Qd_mat"].astype(float)   # (201, 3511)
processed_Vd_mat = data["processed_Vd_mat"].astype(float)   # (201, 3511)

Qmax_all = processed_Qd_mat[-1, :].ravel()
V_mean_rows = processed_Vd_mat.mean(axis=1)   # (n_rows,)

# =============================================================================
# =============================================================================
nature_colors = [
    "#313695",
    "#74add1",
    "#e0f3f8",
    "#fee090",
    "#f46d43",
    "#a50026",
]
cmap_nature = LinearSegmentedColormap.from_list("Nature_Style", nature_colors, N=256)
norm_nature = Normalize(vmin=np.nanmin(Qmax_all), vmax=np.nanmax(Qmax_all))

# =============================================================================
# =============================================================================
Vtail_upp_list = [2.73, 2.75, 2.77]

for Vtail_upp in Vtail_upp_list:
    print(f"Generating plots for Vtail_upp = {Vtail_upp} V...")
    
    Vtail_low = 2.50
    Vseg_low = Vtail_upp
    Vseg_upp = Vtail_upp + 0.30
    nun_seg = 6

    fig = plt.figure(figsize=(24, 6))
    
    gs = fig.add_gridspec(
        nrows=2, ncols=7,
        width_ratios=[1.3, 0.25, 1, 1, 1, 0.03, 0.06], 
        wspace=0.05, hspace=0.08
    )
    
    fig.subplots_adjust(left=0.05, right=0.95, bottom=0.15, top=0.95)

    # -------------------------------------------------------------------------
    # -------------------------------------------------------------------------
    ax_tail = fig.add_subplot(gs[:, 0])

    tail_mask = (V_mean_rows >= Vtail_low) & (V_mean_rows <= Vtail_upp)
    tail_rows = np.where(tail_mask)[0]

    if tail_rows.size > 0:
        Q_tail = processed_Qd_mat[tail_rows, :].T
        valid_tail_mask = np.all(np.isfinite(Q_tail), axis=1) & np.isfinite(Qmax_all)
        Q_tail_valid = Q_tail[valid_tail_mask]
        Qmax_tail    = Qmax_all[valid_tail_mask]

        if Q_tail_valid.shape[0] > 10:
            tsne_tail = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate='auto', init='pca')
            tail_emb = tsne_tail.fit_transform(Q_tail_valid)

            ax_tail.scatter(
                tail_emb[:, 0], tail_emb[:, 1],
                c=Qmax_tail, s=10, cmap=cmap_nature, norm=norm_nature,
                edgecolors="none", alpha=0.9,
            )

            ax_tail.set_xlabel("t-SNE dim 1", fontsize=20, labelpad=15)
            ax_tail.set_ylabel("t-SNE dim 2", fontsize=20, labelpad=15)
            ax_tail.grid(False)
            for spine in ["top", "right", "left", "bottom"]:
                ax_tail.spines[spine].set_visible(True)
            
    # -------------------------------------------------------------------------
    # -------------------------------------------------------------------------
    seg_mask = (V_mean_rows >= Vseg_low) & (V_mean_rows <= Vseg_upp)
    seg_rows_all = np.where(seg_mask)[0]

    ax_shared = None
    if seg_rows_all.size > 0:
        seg_rows_splits = np.array_split(seg_rows_all, nun_seg)

        for k, rows_k in enumerate(seg_rows_splits):
            r = k // 3
            c = (k % 3) + 2
            
            if ax_shared is None:
                ax = fig.add_subplot(gs[r, c])
                ax_shared = ax
            else:
                ax = fig.add_subplot(gs[r, c], sharex=ax_shared, sharey=ax_shared)

            if r == 0:
                ax.tick_params(labelbottom=False)
            if c > 2:
                ax.tick_params(labelleft=False)

            if rows_k.size == 0:
                ax.axis("off")
                continue

            Q_seg = processed_Qd_mat[rows_k, :].T
            valid_seg_mask = np.all(np.isfinite(Q_seg), axis=1) & np.isfinite(Qmax_all)
            Q_seg_valid = Q_seg[valid_seg_mask]
            Qmax_seg    = Qmax_all[valid_seg_mask]

            if Q_seg_valid.shape[0] < 10:
                ax.axis("off")
                continue

            n_valid = Q_seg_valid.shape[0]
            perplexity = min(30, max(5, n_valid // 3))

            tsne_seg = TSNE(n_components=2, random_state=42, perplexity=perplexity, learning_rate='auto', init='pca')
            seg_emb = tsne_seg.fit_transform(Q_seg_valid)

            ax.scatter(
                seg_emb[:, 0], seg_emb[:, 1],
                c=Qmax_seg, s=6, cmap=cmap_nature, norm=norm_nature,
                edgecolors="none", alpha=0.9,
            )

            ax.text(0.02, 0.95, f"Frag{k+1}", transform=ax.transAxes, ha="left", va="top", fontsize=20)
            ax.grid(False)
            for spine in ["top", "right", "left", "bottom"]:
                ax.spines[spine].set_visible(True)

    ax_seg_label = fig.add_subplot(gs[:, 2:5], frameon=False)
    ax_seg_label.tick_params(labelcolor='none', top=False, bottom=False, left=False, right=False)
    ax_seg_label.set_xlabel("t-SNE dim 1", fontsize=20, labelpad=15)
    ax_seg_label.set_ylabel("t-SNE dim 2", fontsize=20, labelpad=15)

    # -------------------------------------------------------------------------
    # -------------------------------------------------------------------------
    cax = fig.add_subplot(gs[:, 6])
    
    sm_global = ScalarMappable(norm=norm_nature, cmap=cmap_nature)
    sm_global.set_array([])
    
    cbar_global = fig.colorbar(sm_global, cax=cax)
    cbar_global.set_label("Maximum capacity (Ah)", fontsize=20, labelpad=15)
    
    cbar_global.outline.set_visible(False)

    # -------------------------------------------------------------------------
    # -------------------------------------------------------------------------
    save_filename = f"combined_tail_and_segments_tsne_Vtail_{Vtail_upp}.png"
    plt.savefig(save_filename, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_filename}\n")

plt.show()